In [245]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [246]:
df= pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

In [247]:
df.head()
df['State'].value_counts()

State
CA    1741433
FL     880192
TX     582837
SC     382557
NY     347960
NC     338199
VA     303301
PA     296620
MN     192084
OR     179660
AZ     170609
GA     169234
IL     168958
TN     167388
MI     162191
LA     149701
NJ     140719
MD     140417
OH     118115
WA     108221
AL     101044
UT      97079
CO      90885
OK      83647
MO      77323
CT      71005
IN      67224
MA      61996
WI      34688
KY      32254
NE      28870
MT      28496
IA      26307
AR      22780
NV      21665
KS      20992
DC      18630
RI      16971
MS      15181
DE      14097
WV      13793
ID      11376
NM      10325
NH      10213
WY       3757
ND       3487
ME       2698
VT        926
SD        289
Name: count, dtype: int64

In [ ]:
df_ia= df[df['State'] == 'IA']

In [ ]:
df_ia.shape

(296620, 46)

In [250]:
columns= ['Severity', 'Start_Time', 'Start_Lat','Start_Lng',  'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)','Precipitation(in)', 'Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']

In [ ]:

df_ia=df_ia[columns]

In [ ]:
df_ia.columns

Index(['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Temperature(F)',
       'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Crossing',
       'Junction', 'Railway', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset'],
      dtype='object')

In [ ]:
df_ia.notnull().sum()

Severity             296620
Start_Time           296620
Start_Lat            296620
Start_Lng            296620
Temperature(F)       290625
Wind_Chill(F)        237281
Humidity(%)          289910
Pressure(in)         291113
Visibility(mi)       289957
Wind_Speed(mph)      273720
Precipitation(in)    226819
Weather_Condition    290596
Crossing             296620
Junction             296620
Railway              296620
Stop                 296620
Traffic_Signal       296620
Sunrise_Sunset       295042
dtype: int64

In [ ]:
df_ia.fillna({
    'Temperature(F)': df_ia['Temperature(F)'].mean(),
    'Humidity(%)': df_ia['Humidity(%)'].mean(),
    'Pressure(in)': df_ia['Pressure(in)'].mean(),
    'Visibility(mi)': df_ia['Visibility(mi)'].mean(),
    'Wind_Speed(mph)': df_ia['Wind_Speed(mph)'].mean(),
    'Precipitation(in)':df_ia['Precipitation(in)'].mean(),
    'Wind_Chill(F)': df_ia['Wind_Chill(F)'].mean()
},inplace=True)

In [ ]:
df_ia.fillna({
    'Weather_Condition': df_ia['Weather_Condition'].mode()[0],
    'Sunrise_Sunset': df_ia['Sunrise_Sunset'].mode()[0]
},inplace=True)

In [ ]:
df_ia['Start_Time'] = pd.to_datetime(df_ia['Start_Time'], errors='coerce')
df_ia['hour']= df_ia['Start_Time'].dt.hour
df_ia['day']= df_ia['Start_Time'].dt.dayofweek
df_ia['month']= df_ia['Start_Time'].dt.month

In [ ]:
df_ia.sample(12)
df_ia.notnull().sum()
df_ia.fillna({
    'hour': df_ia['hour'].mode()[0],
    'day': df_ia['day'].mode()[0],
    'month': df_ia['month'].mode()[0]
}, inplace=True)

In [ ]:
df_ia['is_rush_hour'] = df_ia['hour'].isin(range(7,10)).astype(int) | df_ia['hour'].isin(range(16,20)).astype(int)
df_ia['is_weekend'] = (df_ia['day'] >= 5).astype(int)
df_ia['is_winter'] = df_ia['month'].isin([11,12,1,2]).astype(int)

In [ ]:
df_ia.notnull().sum()
df_ia.drop(columns= ['Start_Time'], inplace=True)

In [ ]:

df_ia.notnull().sum()

Severity             296620
Start_Lat            296620
Start_Lng            296620
Temperature(F)       296620
Wind_Chill(F)        296620
Humidity(%)          296620
Pressure(in)         296620
Visibility(mi)       296620
Wind_Speed(mph)      296620
Precipitation(in)    296620
Weather_Condition    296620
Crossing             296620
Junction             296620
Railway              296620
Stop                 296620
Traffic_Signal       296620
Sunrise_Sunset       296620
hour                 296620
day                  296620
month                296620
is_rush_hour         296620
is_weekend           296620
is_winter            296620
dtype: int64

In [261]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

ohe= OneHotEncoder(handle_unknown='ignore',sparse_output=False)

In [262]:
cat_cols= ['Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
preprocessor= ColumnTransformer(transformers=[
    ('cat', ohe, cat_cols)
], remainder= 'passthrough')   
 

In [ ]:
import numpy as np

df_ia = df_ia.copy()

# remove impossible coordinates
df_ia = df_ia[
    (df_ia['Start_Lat'].between(-90, 90)) &
    (df_ia['Start_Lng'].between(-180, 180))
]

# remove impossible weather values
df_ia = df_ia[df_ia['Temperature(F)'] > -100]   # sanity check
df_ia = df_ia[df_ia['Wind_Speed(mph)'] >= 0]
df_ia = df_ia[df_ia['Pressure(in)'] > 0]
df_ia = df_ia[df_ia['Humidity(%)'].between(0, 100)]
df_ia = df_ia[df_ia['Visibility(mi)'] >= 0]
df_ia = df_ia[df_ia['Precipitation(in)'] >= 0]

df_ia = df_ia[
    (df_ia['Wind_Chill(F)'] > -100) &
    (df_ia['Wind_Chill(F)'] < 150)
]

In [ ]:
def iqr_filter(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return df[(df[col] >= lower) & (df[col] <= upper)]

numeric_cols = [
    'Temperature(F)',
    'Wind_Speed(mph)',
    'Pressure(in)',
    'Visibility(mi)',
    'Humidity(%)',
    'Wind_Chill(F)',
    'Precipitation(in)'
]

for col in numeric_cols:
    df_ia = iqr_filter(df_ia, col)

In [ ]:
lat_min, lat_max = df_ia['Start_Lat'].quantile([0.01, 0.99])
lng_min, lng_max = df_ia['Start_Lng'].quantile([0.01, 0.99])

df_ia = df_ia[
    (df_ia['Start_Lat'].between(lat_min, lat_max)) &
    (df_ia['Start_Lng'].between(lng_min, lng_max))
]

In [ ]:
x= df_ia.drop(columns=['Severity'])
y= df_ia['Severity']


In [ ]:
df_ia['Severity'].value_counts()

Severity
2    173662
3     21076
4     10275
1      1296
Name: count, dtype: int64

In [ ]:
y1=df_ia['Severity']-1

In [269]:
y1.value_counts()

Severity
1    173662
2     21076
3     10275
0      1296
Name: count, dtype: int64

In [ ]:
y1 = (df_ia['Severity'] >= 3).astype(int)

In [271]:
y1.value_counts()

Severity
0    174958
1     31351
Name: count, dtype: int64

In [ ]:
df_ia.shape

(206309, 23)

In [273]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x, y1, test_size=0.2, random_state=42
)


# pipeline
model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    ))
])

# train
model2.fit(x_train, y_train)

# predict (0 or 1)
# y_pred = model2.predict(x_test)
y_prob = model2.predict_proba(x_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

# evaluate
print("Accuracy:", model2.score(x_test, y_test))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8236149483786535

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.83      0.89     34974
           1       0.45      0.79      0.58      6288

    accuracy                           0.82     41262
   macro avg       0.71      0.81      0.73     41262
weighted avg       0.88      0.82      0.84     41262


Confusion Matrix:
 [[29026  5948]
 [ 1330  4958]]


In [274]:
import joblib
joblib.dump(model2,"pa_model.pkl")

['pa_model.pkl']